# IAM Handwriting TrOCR Validation
Validate TrOCR inference on line-level images from the [Teklia/IAM-line](https://huggingface.co/datasets/Teklia/IAM-line) dataset, then pass predictions through Groq for LLM-based correction.

In [ ]:
!pip install datasets transformers torch pillow groq

In [ ]:
from datasets import load_dataset

print("Loading Teklia/IAM-line dataset from Hugging Face...")
ds = load_dataset("Teklia/IAM-line")
print(f"Dataset loaded. Splits: {list(ds.keys())}")
print(f"Train set size: {len(ds['train'])} samples")
print(f"\nSample entry fields: {list(ds['train'].features.keys())}")
print(f"Sample text: '{ds['train'][0]['text']}'")

### Step 1: TrOCR Inference on Line Images

In [ ]:
import torch
from transformers import AutoProcessor, VisionEncoderDecoderModel, logging
from PIL import Image
import warnings

# Suppress warnings for clean output
logging.set_verbosity_error()
warnings.filterwarnings("ignore")

print("Loading TrOCR processor and model...")
processor = AutoProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# Pick 5 sample line images from the training split
NUM_SAMPLES = 5
samples = [ds['train'][i] for i in range(NUM_SAMPLES)]

print(f"\nRunning TrOCR inference on {NUM_SAMPLES} line images...\n")
for i, sample in enumerate(samples):
    image = sample['image'].convert("RGB")
    gt_text = sample['text']

    pixel_values = processor(image, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values, max_new_tokens=64)
    predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print(f"[Sample {i+1}]")
    print(f"  Ground Truth : {gt_text}")
    print(f"  TrOCR Output : {predicted_text}\n")

### Step 2: Word-Level Accuracy Scoring

In [ ]:
# Evaluate word-level accuracy for each line image
# Model and processor are already loaded from Step 1

NUM_EVAL = 5
eval_samples = [ds['train'][i] for i in range(NUM_EVAL)]

reconstructed_sentences_for_groq = []

print(f"Evaluating word-level accuracy on {NUM_EVAL} line images...\n")
for i, sample in enumerate(eval_samples):
    image = sample['image'].convert("RGB")
    gt_text = sample['text']

    # Run TrOCR inference
    pixel_values = processor(image, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values, max_new_tokens=64)
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Split into words for comparison
    gt_words = gt_text.split()
    pred_words = pred_text.split()

    # Compute word-level accuracy (case-insensitive match count / total GT words)
    matches = 0
    for gw, pw in zip(gt_words, pred_words):
        if gw.lower() == pw.lower():
            matches += 1
    accuracy = (matches / len(gt_words)) * 100 if gt_words else 0.0

    print(f"--- Line {i+1} ---")
    print(f"Ground Truth : {gt_text}")
    print(f"Prediction   : {pred_text}")
    print(f"Accuracy     : {matches}/{len(gt_words)} ({accuracy:.1f}%)\n")

    reconstructed_sentences_for_groq.append(pred_text)

### Step 3: LLM Explanation via Groq with Post-Processing

In [ ]:
import os
import difflib
import string
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception as e:
    print(f"Could not retrieve GROQ_API_KEY from Colab secrets: {e}")
    print("Please ensure you have added GROQ_API_KEY to the Secrets tab on the left sidebar.")
    import getpass
    GROQ_API_KEY = getpass.getpass("Alternatively, paste your Groq API Key here: ").strip()
    if not GROQ_API_KEY:
        raise ValueError("API Key cannot be empty. Please provide a valid Groq API Key.")

client = Groq(api_key=GROQ_API_KEY)

def explain_reconstruction(sentence):
    system_prompt = (
        "You are an AI assistant correcting OCR output from handwriting recognition.\n"
        "For each sentence you receive, respond with EXACTLY this format:\n\n"
        "Added content: <YES | NO>\n"
        "Corrected: <your best corrected transcription>\n"
        "Confidence: <HIGH | MEDIUM | LOW>\n"
        "Note: <any notes, or omit this line if confidence is HIGH>\n"
        "Context: <1-2 sentence explanation, ONLY if confidence is HIGH>\n\n"
        "Rules:\n"
        "1. Fix obvious OCR/transcription errors (misspellings, garbled tokens) while preserving the original meaning.\n"
        "2. Before assigning confidence, first answer: \"Did I add, infer, or invent any word, phrase, or meaning not directly present in the OCR input — including completing a sentence fragment, or substituting a different word/phrasing than what was written even if grammatically similar?\" Output this as \"Added content: YES\" or \"Added content: NO\" as the FIRST line of your response, before anything else.\n"
        "3. If Added content = YES, Confidence MUST be MEDIUM or LOW. It cannot be HIGH under any circumstance.\n"
        "4. If Added content = NO, Confidence MAY be HIGH.\n"
        "5. Confidence label definitions:\n"
        "   - HIGH: the corrected sentence clearly reflects the intended meaning, and no words/phrases were added, inferred, or substituted beyond fixing character-level noise (spacing, capitalization, punctuation).\n"
        "   - MEDIUM: some words are uncertain, or minor inference was needed, but the gist is likely correct.\n"
        "   - LOW: more than 2 words were substantially altered, or the corrected sentence's meaning is a guess rather than a clear fix.\n"
        "6. If confidence is MEDIUM or LOW, you MUST include the note: \"This reconstruction may not reflect the original meaning.\" Do NOT provide a contextual explanation paragraph in that case.\n"
        "7. Do NOT invent specific details (times, named actions, first-person narrative, verbs, or events) that are not clearly present in the input tokens.\n"
        "8. Do not rephrase or substitute words that are already correct and clear — only fix genuine OCR noise. Preserve original word choice (\"put down\" must stay \"put down,\" not become \"put forward\").\n"
        "9. Do not output any additional chat or conversational text."
    )
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"OCR output:\n{sentence}"}
        ],
        temperature=0.2,
        max_tokens=300
    )
    return response.choices[0].message.content

def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

print("\n--- Step 3: LLM Explanation via Groq ---\n")
for i, sentence in enumerate(reconstructed_sentences_for_groq, 1):
    print(f"[Line {i}]")
    print(f"  OCR Input : {sentence}")
    
    llm_output = explain_reconstruction(sentence)
    
    # Parse fields
    lines = llm_output.strip().split('\n')
    added_content = "UNKNOWN"
    corrected_line = ""
    confidence = ""
    
    for line in lines:
        if line.startswith("Added content:"):
            added_content = line.split(":", 1)[1].strip()
        elif line.startswith("Corrected:"):
            corrected_line = line.split(":", 1)[1].strip()
        elif line.startswith("Confidence:"):
            confidence = line.split(":", 1)[1].strip()
            
    # Tokenize and compare
    ocr_tokens = tokenize(sentence)
    corrected_tokens = tokenize(corrected_line)
    
    unmatched_tokens = []
    for ct in corrected_tokens:
        matches = difflib.get_close_matches(ct, ocr_tokens, n=1, cutoff=0.7)
        if not matches:
            unmatched_tokens.append(ct)
            
    if len(unmatched_tokens) > 0 and "HIGH" in confidence:
        new_lines = []
        for line in lines:
            if line.startswith("Confidence:"):
                new_lines.append("Confidence: MEDIUM")
            elif line.startswith("Note:") or line.startswith("Context:"):
                continue
            else:
                new_lines.append(line)
                
        while new_lines and not new_lines[-1].strip():
            new_lines.pop()
            
        new_lines.append("Note: This reconstruction may not reflect the original meaning.")
        llm_output = "\n".join(new_lines)
        
    print(f"  Model's Added content: {added_content}")
    print(f"  Computed unmatched tokens: {unmatched_tokens}")
    print(f"  LLM Output:\n{llm_output}\n")
    print("-" * 50)